# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ==============================================================================
# Setup: Libraries, Data Ingestion & Content-Level Aggregation
# ==============================================================================

import duckdb
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import userdata

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score

# 1. Output Directory Configuration
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. DuckDB & Hugging Face Connection Setup
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

FACT_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_PATH  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# 3. Pull Raw Observations (Filtered for Published, Undeleted & GSC Available)
raw_df = con.execute(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_clicks,
        f.gsc_impressions,
        f.ga4_total_engagement_sec,
        c.word_count,
        c.backlinks,
        CASE
            WHEN (f.sessions_ai / (f.gsc_clicks + 1.0) > 0.35) AND (f.sessions_ai >= 5)
            THEN 1 ELSE 0
        END AS is_high_ai_spike
    FROM read_parquet('{FACT_PATH}') f
    JOIN read_parquet('{DIM_PATH}') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE
      AND c.is_published IS TRUE
      AND c.is_deleted IS FALSE
""").df()

# 4. Content-Level Aggregation (Eliminating daily grain duplicates)
frame = raw_df.groupby(["client_hash_id", "content_hash_id"]).agg(
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum"),
    word_count=("word_count", "max"),
    backlinks=("backlinks", "max"),
    is_high_ai_spike=("is_high_ai_spike", "max")
).reset_index()

# 5. Clean Features & Missing Value Indicators
frame["ctr_computed"] = frame["gsc_clicks"] / frame["gsc_impressions"].clip(lower=1.0)
frame["has_word_count"] = frame["word_count"].notnull().astype(int)
frame["word_count_clean"] = frame["word_count"].fillna(0.0)
frame["has_backlinks"] = frame["backlinks"].notnull().astype(int)
frame["backlinks_clean"] = frame["backlinks"].fillna(0.0)
frame["ga4_total_engagement_sec_clean"] = frame["ga4_total_engagement_sec"].fillna(0.0)

# 6. Baseline Rule Score Calculation (from Week 4 Baseline for direct comparison)
ctr_deficit = 1.0 - frame["ctr_computed"].rank(pct=True)
bl_deficit = 1.0 - frame["backlinks_clean"].rank(pct=True)
frame["authority_ctr_deficit"] = (ctr_deficit + bl_deficit) / 2.0
frame["baseline_score"] = np.log1p(frame["word_count_clean"]) * frame["authority_ctr_deficit"]

base_rate = frame["is_high_ai_spike"].mean()
print(f"Dataset successfully prepared: {len(frame):,} content pages across {frame['client_hash_id'].nunique()} unique clients.")
print(f"Content-level base rate (is_high_ai_spike): {base_rate:.4%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset successfully prepared: 176,568 content pages across 46 unique clients.
Content-level base rate (is_high_ai_spike): 0.0583%


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

**Method:** Logistic Regression

**Why it fits the lane:**
* **Question Shape:** Our objective is to identify content patterns in pages with a disproportionately high amount of AI-referred traffic. Since our target (`is_high_ai_spike`) is a binary observed label, Logistic Regression is the most readable and appropriate starting point.
* **Ranking Objective:** The business goal is to rank pages by their probability of being selected/cited by AI, rather than predicting which pages will appear first in traditional search. Logistic Regression allows us to use probability outputs (`predict_proba`) to rank content candidates and evaluate them using Precision@K.
* **Pattern Identification:** A linear, interpretable model provides clear feature weights. This allows us to directly answer *what drives the AI spike* (e.g., structural depth vs. traditional SEO metrics) rather than hiding behind opaque complexity.

In [2]:
# 1. Method Setup
from sklearn.linear_model import LogisticRegression

# Define the features confirmed in the Week 3 Data Contract (excluding the leaky 'sessions_ai' and sub-metrics)
FEATURES = [
    "gsc_clicks",
    "gsc_impressions",
    "ga4_total_engagement_sec",
    "word_count",
    "backlinks"
]

TARGET = "is_high_ai_spike"

# Initialize the model
# (Setting max_iter higher to ensure convergence with our data scale)
model = LogisticRegression(max_iter=1000, random_state=42)

print(f"Model selected: {model.__class__.__name__}")
print(f"Features mapped: {FEATURES}")

Model selected: LogisticRegression
Features mapped: ['gsc_clicks', 'gsc_impressions', 'ga4_total_engagement_sec', 'word_count', 'backlinks']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

**Split Strategy:** 80/20 Train/Test split, grouped by `client_hash_id`.

**Why this is honest:**
* A random 80/20 split would leak data by placing different pages from the same client into both the training and testing sets.
* The model would learn to memorize specific client baseline traffic patterns rather than learning the actual features that drive AI spikes.
* By grouping the 80/20 split on `client_hash_id`, we guarantee that the 20% holdout set consists of entirely unseen clients, proving the model's ability to generalize to new customers.

In [3]:
# 2. Split implementation
from sklearn.model_selection import GroupShuffleSplit

# Using GroupShuffleSplit for an 80/20 split that keeps clients isolated
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

# 'frame' should be your cleaned DataFrame from Week 3/4 containing your features, target, and context columns
train_idx, test_idx = next(gss.split(frame[FEATURES], frame[TARGET], groups=frame['client_hash_id']))

X_train = frame.iloc[train_idx][FEATURES]
y_train = frame.iloc[train_idx][TARGET]

X_test = frame.iloc[test_idx][FEATURES]
y_test = frame.iloc[test_idx][TARGET]

print(f"Training set: {len(X_train)} rows")
print(f"Testing set: {len(X_test)} rows")

Training set: 151339 rows
Testing set: 25229 rows


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

**Comparison Strategy:**
To ensure an honest comparison, both the Week 4 baseline rule and the new Logistic Regression model are evaluated strictly on the 20% test split. We are comparing them using Precision@K (ranking the top 20, 50, 100, 200, and 500 pages) since our ultimate goal is to generate a prioritized queue for content interventions.

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. exact feature names from your Week 3 Data Contract
MODEL_FEATURES = [
    "gsc_clicks",
    "gsc_impressions",
    "ga4_total_engagement_sec",
    "word_count",
    "backlinks"
]

# 2. Handle missing values (Fill NaN with 0)
X_train_clean = X_train[MODEL_FEATURES].fillna(0)
X_test_clean = X_test[MODEL_FEATURES].fillna(0)

# 3. Scale the features (CRITICAL for Logistic Regression with different scales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

# 4. Redefine and train the model with 'balanced' class weights to handle the 0.05% base rate
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# 5. Gather Baseline and Model scores on the TEST split only
test_results = X_test.copy()
test_results["actual_label"] = y_test

# Generate model probabilities for ranking
test_results["model_prob"] = model.predict_proba(X_test_scaled)[:, 1]

# Retrieve the pre-computed baseline score for the test indices
if "baseline_score" in frame.columns:
    test_results["baseline_score"] = frame.loc[test_idx, "baseline_score"]

# 6. Define Precision@K evaluation function
def precision_at_k(df, score_col, label_col, k):
    ranked = df.sort_values(by=score_col, ascending=False)
    return ranked[label_col].iloc[:k].mean()

# 7. Generate the honest comparison table
k_values = [20, 50, 100, 200, 500]
test_base_rate = test_results["actual_label"].mean()
safe_base_rate = test_base_rate if test_base_rate > 0 else 1e-9

comparison_data = []
for k in k_values:
    if k <= len(test_results):
        base_pk = precision_at_k(test_results, "baseline_score", "actual_label", k)
        model_pk = precision_at_k(test_results, "model_prob", "actual_label", k)

        comparison_data.append({
            "K": k,
            "Baseline P@K": base_pk,
            "Model P@K": model_pk,
            "Baseline Lift": base_pk / safe_base_rate,
            "Model Lift": model_pk / safe_base_rate
        })

comparison_df = pd.DataFrame(comparison_data)

print(f"Test Set Base Rate: {test_base_rate:.4%}\n")
print("=== Honest Comparison: Baseline vs. Logistic Regression ===")
print(comparison_df.to_markdown(index=False, floatfmt=".4f"))

Test Set Base Rate: 0.0595%

=== Honest Comparison: Baseline vs. Logistic Regression ===
|        K |   Baseline P@K |   Model P@K |   Baseline Lift |   Model Lift |
|---------:|---------------:|------------:|----------------:|-------------:|
|  20.0000 |         0.0000 |      0.0000 |          0.0000 |       0.0000 |
|  50.0000 |         0.0000 |      0.0000 |          0.0000 |       0.0000 |
| 100.0000 |         0.0000 |      0.0100 |          0.0000 |      16.8193 |
| 200.0000 |         0.0000 |      0.0050 |          0.0000 |       8.4097 |
| 500.0000 |         0.0000 |      0.0020 |          0.0000 |       3.3639 |


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# Logistic Regression coefficients to see what the model leaned on
coef_df = pd.DataFrame({
    'Feature': MODEL_FEATURES,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print("=== Logistic Regression Feature Weights ===")
print(coef_df.to_markdown(index=False))

=== Logistic Regression Feature Weights ===
| Feature                  |   Coefficient |
|:-------------------------|--------------:|
| word_count               |     1.07779   |
| ga4_total_engagement_sec |     0.889062  |
| gsc_impressions          |     0.621736  |
| gsc_clicks               |    -0.0240231 |
| backlinks                |    -3.9801    |


## 4. Errors and interpretation

**Where is the model wrong?**
The model suffers from an extreme false-positive rate. Even though it achieves a ~16.8x lift over the base rate at K=100, its absolute Precision@100 is only 1.00%. Because the base rate of an AI spike is so incredibly low (0.059%), predicting it with only five blunt structural/search features results in the model flagging dozens of "good" pages for every true spike it finds.

**What does it lean on?**
* **Penalizing Traditional Authority:** The strongest signal is a massive negative weight on `backlinks` (-3.98). The model learned that pages experiencing sudden "AI spikes" are usually *not* the established, high-authority pages dominating traditional SEO.
* **Rewarding Depth:** It leans heavily on `word_count` (+1.07) and `ga4_total_engagement_sec` (+0.88). It looks for deep, engaging content that LLMs love to parse and cite.
* **The Impression Trap:** It positively weights `gsc_impressions` (+0.62). This is likely a primary driver of the model's false positives, as it accidentally rewards highly visible, mainstream pages rather than focusing strictly on hidden, high-depth informational gems.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.